# Fine-tune mT5-small — Uzbek Text Simplifier (капстоун)

Датасеты: `output.csv`, `output2.csv` (text/simplified_text), `domain_legal.csv` (text/text_simplified).

**Runtime → T4 GPU** (mT5-small лёгкая — хватит и free tier)

In [ ]:
!pip install -q transformers datasets evaluate sentencepiece accelerate rouge_score sacrebleu

## 1. Загрузка файлов

In [ ]:
from google.colab import files
import os

needed = ['output.csv', 'output2.csv', 'domain_legal.csv']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print('Загрузи файлы:', missing)
    uploaded = files.upload()

In [ ]:
import pandas as pd

df1 = pd.read_csv('output.csv')[['text', 'simplified_text']]
df2 = pd.read_csv('output2.csv')[['text', 'simplified_text']]
df3 = pd.read_csv('domain_legal.csv').rename(columns={'text_simplified': 'simplified_text'})[['text', 'simplified_text']]

frames = [df1, df2, df3]
if os.path.exists('uz_simplified.csv'):
    df4 = pd.read_csv('uz_simplified.csv').rename(columns={'simplified': 'simplified_text'})[['text', 'simplified_text']]
    frames.append(df4)
    print('uz_simplified.csv:', len(df4), 'строк добавлено')

df = pd.concat(frames, ignore_index=True)
df = df.dropna(subset=['text', 'simplified_text'])
df = df[df['text'].str.strip() != '']
df = df[df['simplified_text'].str.strip() != '']
df = df.drop_duplicates(subset=['text'])

# mT5 контекст скромный на практике — отсекаем экстремально длинные примеры
MAX_CHARS = 3000
df = df[(df['text'].str.len() <= MAX_CHARS) & (df['simplified_text'].str.len() <= MAX_CHARS)]

print('Итого примеров:', len(df))
df.head(2)

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
})
print(ds)

## 2. Модель и токенизация

T5-семейство требует task-префикс — используем `"simplify: "`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/mt5-small"
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 256
PREFIX = "simplify: "

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

In [ ]:
def preprocess(batch):
    inputs = [PREFIX + t for t in batch['text']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch['simplified_text'], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)

## 3. Метрики (ROUGE + chrF — chrF надёжнее ROUGE на агглютинативных языках вроде узбекского)

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load('rouge')
chrf = evaluate.load('chrf')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    chrf_result = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result['chrf'] = chrf_result['score']
    return {k: round(v, 4) if isinstance(v, float) else v for k, v in result.items()}

## 4. Обучение

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import torch

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir='mt5_uz_simplify',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_ratio=0.05,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='chrf',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 5. Быстрый тест

In [ ]:
def simplify(text, max_new_tokens=200):
    inputs = tokenizer(PREFIX + text, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample = val_df.iloc[0]['text']
print('ORIGINAL:', sample[:300])
print()
print('SIMPLIFIED:', simplify(sample))
print()
print('REFERENCE:', val_df.iloc[0]['simplified_text'][:300])

## 6. Сохранение и скачивание

In [ ]:
trainer.save_model('mt5_uz_simplify_final')
tokenizer.save_pretrained('mt5_uz_simplify_final')

!zip -r mt5_uz_simplify_final.zip mt5_uz_simplify_final
from google.colab import files as colab_files
colab_files.download('mt5_uz_simplify_final.zip')

## 7. (опционально) Пуш на HuggingFace Hub — для FastAPI/Gradio деплоя

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub('your-username/mt5-uz-simplify')